In [ ]:
#@title **_Installing_ dan _Importing Library_**
!pip install -U scikit-learn
!pip install Sastrawi

import nltk
import re
import numpy as np
import pandas as pd

import Sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.corpus import wordnet
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
#@title **_Mount Google Drive_** # Jika tidak menggunakan Google Colab, bisa dihapus/abaikan
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title **Membaca File Ulasan**
df = pd.read_csv('your_file.csv') # Ubah your_file.csv dengan nama file yang ingin diolah
df

In [ ]:
#@title **Hapus kolom yang tidak diperlukan**
df = df.drop(['column'], axis=1)
df.head(10)

In [ ]:
#@title **_Casefolding_ dan _cleansing_**
def preprocess(text):
    text = text.lower()

    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\b\w*([a-z])\1{1}\w*\b', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['content'] = df['content'].apply(lambda x: preprocess(x))
df.head(10)

In [ ]:
#@title **Normalisasi kata**
normalisasi_kata = {
    "bhs" : "bahasa","gak": "tidak","ga": "tidak","gk": "tidak","tdk": "tidak",
    "gw": "saya","gue": "saya","gwe": "saya","bgt": "banget","bngt": "banget",
    "make": "pakai","pake": "pakai","moga": "semoga","mayan": "lumayan","yg": "yang",
    "nambah": "tambah","gitu": "begitu","muas": "puas","pdhl": "padahal","ngerjain": "kerjain",
    "hp": "handphone","nggak": "tidak","males": "malas","rame": "ramai","kocak": "lucu",
    "mabar": "main bareng","nyebelin": "menyebalkan","update": "pembaruan","ngebug": "bug",
    "ngulang": "mengulang","ngalamin": "mengalami","ngetes": "mengujicoba","ngejar": "mengejar"
}

def normalize_text(text):
    words = text.split()
    normalized_words = [normalisasi_kata.get(word, word) for word in words]
    return ' '.join(normalized_words)


df['content'] = df['content'].apply(normalize_text)
df.head(10)

In [ ]:
#@title **Tokenisasi**
df['content'] = df['content'].apply(word_tokenize)
df.head(10)

In [ ]:
#@title **_Stopword Removal_**
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Membuat instance stopword factory dari Sastrawi
factory = StopWordRemoverFactory()

# Mendapatkan daftar stopword dari Sastrawi
stopword = factory.get_stop_words()

# Daftar kata-kata negasi yang ingin dipertahankan
kata_negasi = ["tidak", "belum", "tanpa", "nggak", "melainkan"]

# Menghapus kata negasi dari daftar stopword
stopword = [word for word in stopword if word not in kata_negasi]

# Fungsi untuk menghapus stopword dari teks
def remove_stopwords(text):
    return [word for word in text if word not in stopword]

# Menerapkan fungsi stopword removal pada kolom 'content' di dataframe
df['content'] = df['content'].apply(remove_stopwords)

# Menampilkan 10 baris pertama hasilnya
df.head(10)


In [ ]:
#@title **_Stemming_ menggunakan Sastrawi**
factory = StemmerFactory()
stemmer = factory.create_stemmer()
df['content'] = df['content'].apply(lambda x: [stemmer.stem(word) for word in x])
df['content'] = df['content'].apply(lambda x: ' '.join(x))
df.head(10)

In [ ]:
#@title **Menghapus data kosong**
df = df[df['content'] != '']
df.head(10)

In [ ]:
#@title **Menyimpan ulasan yang telah melalui pra-pemrosesan teks**
df.to_csv('preprocessed.csv', index=False)

In [ ]:
#@title **Membaca data ulasan yang telah melalui pra-pemrosesan teks**
df = pd.read_csv('preprocessed.csv')

In [ ]:
#@title **_Labeling_ Menggunakan _Modified Indonesia Sentiment Lexicon_**

inset = pd.read_csv('lexicon.csv', encoding='latin-1')

lexicon = dict(zip(inset['word'], inset['weight']))

def get_sentiment(text):
    sentiment_score = 0
    words = text.split()
    for word in words:
        if word in lexicon:
            sentiment_score += lexicon[word]
    return sentiment_score

df['label_score'] = df['content'].apply(get_sentiment)

def categorize_sentiment(score):
    if score > 0:
        return 'positif'
    elif score < 0:
        return 'negatif'
    else:
        return 'netral'

df['label'] = df['label_score'].apply(categorize_sentiment)

df.head(10)

In [ ]:
#@title Menyimpan _dataset_ yang telah dilabeli
df.to_csv('labeled.csv', index=False)